In [1]:
from pathlib import Path
print(Path("./").resolve())

import os
import importlib
from pydantic import AwareDatetime, BaseModel
from typing import *
from enum import StrEnum
from zoneinfo import ZoneInfo
from datetime import datetime, timedelta, timezone, date
from pathlib import Path
from tqdm import tqdm
from itertools import chain, islice
from collections import defaultdict
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import pandas as pd
import yaml
import json
import requests
import pickle
import configparser
import nsapi

C:\Users\jeffw\Code\IDEA\dutch_railways_server\database\src\scraping


In [2]:
working_dir = "./"

In [3]:
# Parameters
working_dir = "C:\\Users\\jeffw\\Code\\IDEA\\dutch_railways_server\\database\\build\\scraping"


In [4]:
out_dir = Path(working_dir) / 'ns_results'
raw_out_dir = out_dir / 'raw'
out_dir.mkdir(parents=True, exist_ok=True)
raw_out_dir.mkdir(parents=True, exist_ok=True)

In [5]:
NS_PRIMARY_KEY = os.environ["NS_PRIMARY_KEY"]
NS_SECONDARY_KEY = os.environ["NS_SECONDARY_KEY"]

NS_GET_HEADER = {
    'Ocp-Apim-Subscription-Key': NS_PRIMARY_KEY,
}

# https://apiportal.ns.nl/apis
NSAPI_URLS = {}
with open('./nsapi/_nsapp-stations-api.yaml') as file:
    NSAPI_URLS['STATIONS']       = yaml.full_load(file)['servers'][0]['url']
with open('./nsapi/_reisinformatie-api.yaml') as file:
    NSAPI_URLS['REISINFORMATIE'] = yaml.full_load(file)['servers'][0]['url']
with open('./nsapi/_virtual-train-API.yaml') as file:
    NSAPI_URLS['VIRTUAL_TRAIN']  = yaml.full_load(file)['servers'][0]['url']

In [6]:
# below not useful (only works with API at foreign station departures)
# FOR_DATE: AwareDatetime = datetime(year=2026, month=6, day=1, tzinfo=ZoneInfo('Europe/Amsterdam'))

## Get train trips

In [7]:
# VEHICLE '/vehicle' https://apiportal.ns.nl/api-details#api=virtual-train-API&operation=getVehicles
# treinen[].ritId (string)
# treinen[].richting (float)
# treinen[].type (string)

In [8]:
vehicles_pkl = raw_out_dir / 'vehicle.pkl'
if not vehicles_pkl.exists():
    response = requests.get(NSAPI_URLS['VIRTUAL_TRAIN'] + '/vehicle', headers=NS_GET_HEADER,
        params=nsapi.virtual_train.ApiVehicleGetParameters(
            lat=52.366667, long=4.883333,
            radius=300000,
            # limit=None,
            # route=route_number,
            # features=materieel_value,
        )
    )
    print('Response:', response.json())
    obj = nsapi.virtual_train.Treinen(**response.json()['payload'])
    with open(vehicles_pkl, 'wb') as file:
        pickle.dump(obj, file, protocol=-1)
else:
    with open(vehicles_pkl, 'rb') as file:
        obj = pickle.load(file)

Response: {'payload': {'treinen': [{'treinNummer': 8758, 'ritId': '8758', 'lat': 52.077145, 'lng': 4.6467066, 'snelheid': 0.06, 'richting': 5.67, 'horizontaleNauwkeurigheid': 280272.22, 'type': 'SPR', 'bron': 'OBIS'}, {'treinNummer': 6461, 'ritId': '6461', 'lat': 51.329002, 'lng': 5.611201, 'snelheid': 139.92, 'richting': 154.06, 'horizontaleNauwkeurigheid': 280233.34, 'type': 'SPR', 'bron': 'OBIS'}, {'treinNummer': 6561, 'ritId': '6561', 'lat': 51.45571, 'lng': 5.7896214, 'snelheid': 0.0, 'richting': 105.62, 'horizontaleNauwkeurigheid': 280155.56, 'type': 'SPR', 'bron': 'OBIS'}, {'treinNummer': 6669, 'ritId': '6669', 'lat': 51.95502, 'lng': 5.851888, 'snelheid': 44.85, 'richting': 176.95, 'horizontaleNauwkeurigheid': 280427.78, 'type': 'SPR', 'bron': 'OBIS'}, {'treinNummer': 7663, 'ritId': '7663', 'lat': 51.987793, 'lng': 5.9434123, 'snelheid': 0.0, 'richting': 58.41, 'horizontaleNauwkeurigheid': 280350.0, 'type': 'SPR', 'bron': 'OBIS'}, {'treinNummer': 6565, 'ritId': '6565', 'lat': 5

In [9]:
_trains_raw = {
    x.ritId : x.type
    for x in obj.treinen
}
_service_types = Counter(_trains_raw.values())
TrainServiceType = StrEnum('TrainServiceType', ((x.upper(), x) for x in _service_types))

trains_type_by_id: dict[int, TrainServiceType] = {
    int(k) : TrainServiceType(v)
    for k, v in _trains_raw.items()
}

_sorted_ids = sorted(trains_type_by_id.keys())
print(_service_types)
print(f"{ len(_sorted_ids) }: { min(_sorted_ids) } - { max(_sorted_ids) }")

Counter({'IC': 158, 'SPR': 138})
296: 559 - 704364


## Get trip timetables

In [10]:
# JOURNEY DETAILS '/v2/journey' https://apiportal.ns.nl/api-details#api=reisinformatie-api&operation=getJourneyDetail
# payload.productNumbers (string[])
# payload.plannedStock.trainParts[].stockIdentifier (string) # each trainPart has its own destination (see stop)
# payload.plannedStock.trainType (string)
# payload.stops[].stop.UICCode (string)
# payload.stops[].stop.namen.kort/lang/middel (string)
# payload.stops[].arrivals/departures.product.displayName/operatorName

### For single trip

In [11]:
sample_train_id = 9571 #trains_type_by_id.keys().__iter__().__next__()

response = requests.get(NSAPI_URLS['REISINFORMATIE'] + '/api/v2' + '/journey', headers=NS_GET_HEADER,
    params=nsapi.reisinformatie.ApiV2JourneyGetParameters(
        train=sample_train_id, # required when not giving a journey ID
        # dateTime=FOR_DATE, # date only
        omitCrowdForecast=True,
    )
)
obj = nsapi.reisinformatie.RepresentationResponseJourney(**response.json())

class Stop(BaseModel):
    station_uic: int
    name: str
    arrives: AwareDatetime | None
    departs: AwareDatetime | None
class _TrainService0(BaseModel):
    ritnummer: int
    stops: list[Stop]
    categoryCode: str
    categoryName: str
    trainType: str | None
    trainPart_facilities: dict[int, list[str]] | None

def product_from_stops(stops: list[nsapi.reisinformatie.JourneyStop]) -> nsapi.reisinformatie.ProductInterface:
    for stop in stops:
        if stop.departures:
            return stop.departures[0].product
        if stop.arrivals:
            return stop.arrivals[0].product
    raise Exception()

def stock_from_stops(stops: list[nsapi.reisinformatie.JourneyStop]) -> nsapi.reisinformatie.Stock:
    for stop in stops:
        if stop.plannedStock:
            return stop.plannedStock
    return None

def extract_journey_data(ritnummer: int, obj: nsapi.reisinformatie.RepresentationResponseJourney):
    obj.payload.stops = [x for x in obj.payload.stops if x.status != "PASSING"]

    # assert (len(set(int(x) for x in obj.payload.productNumbers)) == 1) # not true
    assert str(ritnummer) in obj.payload.productNumbers

    for stop in obj.payload.stops:
        if len(stop.arrivals) > 1 or len(stop.departures) > 1:
            raise Exception('a')
        is_first_stop: bool = not stop.previousStopId
        is_last_stop: bool = not stop.nextStopId
        if not(is_first_stop or len(stop.arrivals) > 0):
            # raise Exception(f'Ensure requested train timetable is not for a train that has already departed. prevStopId: {stop.previousStopId}')
            pass # we cannot change this circumstance
        # assert (is_last_stop or len(stop.departures) > 0), stop.nextStopId

        for arr_or_depart in chain(stop.arrivals, stop.departures):
            assert(arr_or_depart.plannedTime.utcoffset() == timedelta(hours=2)) # 1 if DST not in effect

    return _TrainService0(
        ritnummer= ritnummer,
        stops= (Stop(
            station_uic = int(x.stop.uicCode),
            name= x.stop.name,
            arrives= x.arrivals[0].plannedTime if x.arrivals else None,
            departs= x.departures[0].plannedTime if x.departures else None,
        )   for x in obj.payload.stops),
        categoryCode= product_from_stops(obj.payload.stops).categoryCode,
        categoryName= product_from_stops(obj.payload.stops).longCategoryName,
        trainType= stock_from_stops(obj.payload.stops).trainType if stock_from_stops(obj.payload.stops) else None,
        trainPart_facilities= {
            int(y.stockIdentifier): y.facilities
            for y in stock_from_stops(obj.payload.stops).trainParts
        }   if stock_from_stops(obj.payload.stops) else None,
    )

sample_train_timetable = extract_journey_data(sample_train_id, obj)
sample_train_timetable

_TrainService0(ritnummer=9571, stops=[Stop(station_uic=8814001, name='Brussel-Zuid', arrives=None, departs=None), Stop(station_uic=8813003, name='Brussel-Centraal', arrives=None, departs=None), Stop(station_uic=8812005, name='Brussel-Noord', arrives=None, departs=None), Stop(station_uic=8822004, name='Mechelen', arrives=None, departs=None), Stop(station_uic=8821121, name='Antwerpen-Berchem', arrives=None, departs=None), Stop(station_uic=8821006, name='Antwerpen-Centraal', arrives=None, departs=None), Stop(station_uic=8821063, name='Antwerpen-Luchtbal', arrives=None, departs=None), Stop(station_uic=8821105, name='Noorderkempen', arrives=None, departs=None), Stop(station_uic=8400542, name='Rotterdam Lombardijen', arrives=None, departs=None), Stop(station_uic=8400534, name='Rotterdam Stadion', arrives=None, departs=None), Stop(station_uic=8400533, name='Rotterdam Zuid', arrives=None, departs=None), Stop(station_uic=8400529, name='Rotterdam Blaak', arrives=None, departs=None), Stop(station

### For all known train trips

In [12]:
journeys_raw_pkl = raw_out_dir / 'journey_details_raw.pkl'
journeys_pkl = raw_out_dir / 'journey_details.pkl'

def _journey_api_request(train_id: int) -> dict[str, Any]:
    response = requests.get(NSAPI_URLS['REISINFORMATIE'] + '/api/v2' + '/journey', headers=NS_GET_HEADER,
        params=nsapi.reisinformatie.ApiV2JourneyGetParameters(
            train=train_id, # required when not giving a journey ID
            omitCrowdForecast=True,
        )
    )
    return response.json()
def _get_journey_raw_obj() -> dict[str, dict[str, Any]]:
    if not journeys_raw_pkl.exists():
        train_ids = trains_type_by_id.keys()
        raw_obj = {
            train_id : _journey_api_request(train_id)
            for train_id in tqdm(train_ids)
        }
        with open(journeys_raw_pkl, 'wb') as file:
            pickle.dump(raw_obj, file, protocol=-1)
    else:
        with open(journeys_raw_pkl, 'rb') as file:
            raw_obj = pickle.load(file)
    return raw_obj
def _get_journey_obj() -> dict[int, nsapi.reisinformatie.RepresentationResponseJourney]:
    '''May not contain all requested train_ids; some will not be found.'''
    if not journeys_pkl.exists():
        raw_obj = _get_journey_raw_obj()
        obj = dict[int, nsapi.reisinformatie.RepresentationResponseJourney]()
        for k, v in raw_obj.items():
            if (v.get('code', 200) == 404):
                continue
            obj |= {
                k : nsapi.reisinformatie.RepresentationResponseJourney(**v)
            }

        with open(journeys_pkl, 'wb') as file:
            pickle.dump(obj, file, protocol=-1)
    else:
        with open(journeys_pkl, 'rb') as file:
            obj = pickle.load(file)
    return obj

obj = _get_journey_obj()

_trains_timetables: dict[int, _TrainService0] = {
    train_id : extract_journey_data(train_id, journey_obj)
    for train_id, journey_obj in obj.items()
}

_trains_timetables

  0%|          | 0/296 [00:00<?, ?it/s]

  0%|          | 1/296 [00:00<04:09,  1.18it/s]

  1%|          | 2/296 [00:01<03:47,  1.29it/s]

  1%|          | 3/296 [00:02<04:03,  1.20it/s]

  1%|▏         | 4/296 [00:02<03:14,  1.50it/s]

...

 99%|█████████▉| 294/296 [03:47<00:01,  1.34it/s]

100%|█████████▉| 295/296 [03:48<00:00,  1.33it/s]

100%|██████████| 296/296 [03:48<00:00,  1.44it/s]

100%|██████████| 296/296 [03:48<00:00,  1.29it/s]

{8758: _TrainService0(ritnummer=8758, stops=[Stop(station_uic=8400258, name='Gouda', arrives=None, departs=datetime.datetime(2026, 8, 10, 18, 13, tzinfo=TzInfo(7200))), Stop(station_uic=8400677, name='Waddinxveen Triangel', arrives=datetime.datetime(2026, 8, 10, 18, 19, tzinfo=TzInfo(7200)), departs=datetime.datetime(2026, 8, 10, 18, 19, tzinfo=TzInfo(7200))), Stop(station_uic=8400675, name='Waddinxveen', arrives=datetime.datetime(2026, 8, 10, 18, 21, tzinfo=TzInfo(7200)), departs=datetime.datetime(2026, 8, 10, 18, 23, tzinfo=TzInfo(7200))), Stop(station_uic=8400674, name='Waddinxveen Noord', arrives=datetime.datetime(2026, 8, 10, 18, 25, tzinfo=TzInfo(7200)), departs=datetime.datetime(2026, 8, 10, 18, 25, tzinfo=TzInfo(7200))), Stop(station_uic=8400126, name='Boskoop Snijdelwijk', arrives=datetime.datetime(2026, 8, 10, 18, 27, tzinfo=TzInfo(7200)), departs=datetime.datetime(2026, 8, 10, 18, 27, tzinfo=TzInfo(7200))), Stop(station_uic=8400125, name='Boskoop', arrives=datetime.datetime(

#### Create enums

In [13]:
_category_codes = Counter(x.categoryCode for x in _trains_timetables.values())
CategoryCode = StrEnum('CategoryCode', ((x.upper(), x) for x in _category_codes))
print(_category_codes)

_category_names = Counter(x.categoryName for x in _trains_timetables.values())
CategoryName = StrEnum('CategoryName', ((x.upper(), x) for x in _category_names))
print(_category_names)

_train_types = Counter(x.trainType for x in _trains_timetables.values() if x.trainType)
TrainType = StrEnum('TrainType', ((x.upper(), x) for x in _train_types))
print(_train_types)

_facilities = Counter(
    z
    for x in _trains_timetables.values()
    if x.trainPart_facilities
    for y in x.trainPart_facilities.values()
    for z in y
)
Facility = StrEnum('Facility', ((x.upper(), x) for x in _facilities))
print(_facilities)

Counter({'IC': 144, 'SPR': 138, 'ICD': 10, 'ECD': 4})
Counter({'Intercity': 144, 'Sprinter': 138, 'Intercity direct': 10, 'Eurocity Direct': 4})
Counter({'VIRM': 96, 'SNG': 58, 'SLT': 54, 'Flirt': 25, 'DDZ': 22, 'ICNG': 20, 'ICM': 16})
Counter({'TOILET': 373, 'FIETS': 373, 'WIFI': 299, 'STROOM': 269, 'TOEGANKELIJK': 212, 'STILTE': 188})


#### Apply enums

In [14]:
class TrainService(BaseModel):
    ritnummer: int
    stops: list[Stop]
    categoryCode: CategoryCode
    categoryName: CategoryName
    trainType: TrainType | None
class StockFacility(BaseModel):
    trainType: TrainType
    facilities: list[Facility]

train_timetables = {
    k : TrainService(
        ritnummer= v.ritnummer,
        stops= v.stops,
        categoryCode= CategoryCode(v.categoryCode),
        categoryName= CategoryName(v.categoryName),
        trainType= TrainType(v.trainType) if v.trainType else None,
    )
    for k, v in _trains_timetables.items()
}

_facilities_by_trainType: defaultdict[TrainType, Counter[Facility]] = defaultdict(lambda: Counter[Facility]())
for v in _trains_timetables.values():
    if not(v.trainType) and not(v.trainPart_facilities):
        continue
    _facilities_by_trainType[v.trainType].update(
        Facility(x)
        for facilities in v.trainPart_facilities.values()
        for x in facilities
    )
print(json.dumps(_facilities_by_trainType, indent=4))

facilities_by_trainType: dict[TrainType, list[Facility]] = {
    k : list[Facility](v.keys())
    for k, v in _facilities_by_trainType.items()
}

{
    "Flirt": {
        "TOILET": 28,
        "STROOM": 28,
        "FIETS": 28,
        "TOEGANKELIJK": 28,
        "WIFI": 28
    },
    "SNG": {
        "FIETS": 83,
        "WIFI": 83,
        "STROOM": 83,
        "TOEGANKELIJK": 83,
        "TOILET": 83
    },
    "SLT": {
        "FIETS": 74,
        "TOILET": 74,
        "TOEGANKELIJK": 74
    },
    "ICNG": {
        "WIFI": 27,
        "TOILET": 27,
        "STILTE": 27,
        "STROOM": 27,
        "FIETS": 27,
        "TOEGANKELIJK": 27
    },
    "ICM": {
        "WIFI": 27,
        "TOILET": 27,
        "STILTE": 27,
        "STROOM": 27,
        "FIETS": 27
    },
    "DDZ": {
        "WIFI": 25,
        "TOILET": 25,
        "STILTE": 25,
        "STROOM": 25,
        "FIETS": 25
    },
    "VIRM": {
        "WIFI": 109,
        "TOILET": 109,
        "STILTE": 109,
        "FIETS": 109,
        "STROOM": 79
    }
}


## Get train stations

In [15]:
# STATIONS '/v3' https://apiportal.ns.nl/api-details#api=nsapp-stations-api&operation=getStationsV3
# payload.id.uicCode
# payload.id.code # eg UT for Utrecht Centraal
# payload.stationType (enum)
# payload.names.long/medium/short/festive/synonyms[]
# payload.location.lat/lng (number)

In [16]:
stations_pkl = raw_out_dir / 'stations.pkl'
if not stations_pkl.exists():
    response = requests.get(NSAPI_URLS['STATIONS'] + '/v3', headers=NS_GET_HEADER,
        params=nsapi.nsapp_stations.V3GetParameters(
            countryCodes=['NL'],
            limit=None # irrelevant when not providing str `q`
        )
    )
    obj = nsapi.nsapp_stations.StationsV3Response(**response.json())
    with open(stations_pkl, 'wb') as file:
        pickle.dump(obj, file, protocol=-1)
else:
    with open(stations_pkl, 'rb') as file:
        obj = pickle.load(file)

In [17]:
StationInfo = NamedTuple('StationInfo', (
    ('name', str),
    ('lat', float),
    ('lng', float),
))

stations_by_uicCode: dict[int, StationInfo] = {
    int(x.id.uicCode) : StationInfo(
        name=x.names.long,
        lat=x.location.lat,
        lng=x.location.lng,
    )
    for x in obj.payload
}

_uics = sorted(stations_by_uicCode.keys())
_names_len = sorted(
    (x.name for x in stations_by_uicCode.values()),
    key=len
)
_names_words = sorted(
    (x.name for x in stations_by_uicCode.values()),
    key=lambda s: len(s.split())
)
print(_uics[:1] + ['...'] + _uics[-1:])
print('========')
print(_names_len[:3] + ['...'] + _names_len[-3:])
print('========')
print(_names_words[:3] + ['...'] + _names_words[-3:])

[8400045, '...', 8400752]
['Oss', 'Olst', 'Elst', '...', 'Bovenkarspel-Grootebroek', 'Lansingerland-Zoetermeer', 'Leeuwarden Camminghaburen']
["'s-Hertogenbosch", 'Alkmaar', 'Almelo', '...', 'Zandvoort aan Zee', 'Koog aan de Zaan', 'Den Haag Laan v NOI']


## Export for SQL

In [18]:
reveal_type(trains_type_by_id)
reveal_type(train_timetables)
reveal_type(facilities_by_trainType)
reveal_type(stations_by_uicCode)

Runtime type is 'dict'
Runtime type is 'dict'
Runtime type is 'dict'
Runtime type is 'dict'


{8400301: StationInfo(name='Heerenveen IJsstadion', lat=52.9352760314941, lng=5.94388866424561),
 8400534: StationInfo(name='Rotterdam Stadion', lat=51.8938903808594, lng=4.51972198486328),
 8400058: StationInfo(name='Amsterdam Centraal', lat=52.3788871765137, lng=4.90027761459351),
 8400282: StationInfo(name='Den Haag Centraal', lat=52.0802764892578, lng=4.32499980926514),
 8400206: StationInfo(name='Eindhoven Centraal', lat=51.4433326721191, lng=5.48138904571533),
 8400530: StationInfo(name='Rotterdam Centraal', lat=51.9249992370605, lng=4.46888875961304),
 8400561: StationInfo(name='Schiphol Airport', lat=52.3094444274902, lng=4.76194429397583),
 8400621: StationInfo(name='Utrecht Centraal', lat=52.0888900756836, lng=5.11027765274048),
 8400319: StationInfo(name="'s-Hertogenbosch", lat=51.69048, lng=5.29362),
 8400050: StationInfo(name='Alkmaar', lat=52.6377792358398, lng=4.73972225189209),
 8400051: StationInfo(name='Almelo', lat=52.3580551147461, lng=6.65388870239258),
 8400080: S

In [19]:
geolocator = Nominatim(user_agent="ns_train_stations_scraper")
geolocator_rate_limited = RateLimiter(geolocator.reverse, min_delay_seconds=2)

In [20]:
addresses_pkl = raw_out_dir / 'addresses.pkl'
if not addresses_pkl.exists():
    sql_stations = pd.DataFrame({
        'uic': uic,
        'name': v.name,
        'lat': v.lat,
        'lng': v.lng,
        'address': geolocator_rate_limited(f"{v.lat}, {v.lng}").address,
    }   for uic, v in stations_by_uicCode.items())
    with open(addresses_pkl, 'xb') as file:
        pickle.dump(sql_stations, file, protocol=-1)
else:
    with open(addresses_pkl, 'r+b') as file:
        sql_stations = pickle.load(file)

In [21]:
sql_trainsetamenities = pd.DataFrame([
    {
        'trainset': trainset,
        'amenity': amenity,
    }
    for trainset, facilities in facilities_by_trainType.items()
    for amenity in facilities
])
sql_trainsetamenities

,trainset,amenity
0,Flirt,TOILET
1,Flirt,STROOM
2,Flirt,FIETS
3,Flirt,TOEGANKELIJK
4,Flirt,WIFI
5,SNG,FIETS
6,SNG,WIFI
7,SNG,STROOM
8,SNG,TOEGANKELIJK
9,SNG,TOILET


In [22]:
sql_passservice = pd.DataFrame([
    {
        'num': k,
        'name': f"{v.categoryName} {v.ritnummer} to {v.stops[-1].name}",
        'trainset': v.trainType if v.trainType
                    else TrainType.SLT if v.ritnummer == 9571
                    else TrainType.ICNG if v.categoryCode == CategoryCode.ECD
                    else None,
    }
    for k, v in train_timetables.items()
])

In [23]:
sql_stop = pd.DataFrame([
    {
        'passservice_num': passservice_id,
        'arrival': stop.arrives,
        'departure': stop.departs,
        'stations_uic': stop.station_uic,
    }
    for passservice_id, passservice in train_timetables.items()
    for stop in passservice.stops
])

## Final cleanup/asserts

In [24]:
# delete rows missing both arrival/departure
sql_stop = sql_stop[~sql_stop[['arrival', 'departure']].isna().all(axis='columns')]

In [25]:
sql_stop.loc[sql_stop['arrival'] == sql_stop['departure'], 'departure'] += timedelta(seconds=30)
sql_stop.loc[sql_stop['arrival'] == sql_stop['departure']]

,passservice_num,arrival,departure,stations_uic


In [26]:
# double-check, in sql_stop, that only first and last stops are missing arrival/departure
stops_sorted_groups = sql_stop.sort_values('arrival', na_position='first').groupby('passservice_num')
heads_departures = stops_sorted_groups.apply(lambda group: group['departure'].iloc[:-1])
tails_arrivals = stops_sorted_groups.apply(lambda group: group['arrival'].iloc[1:])
assert(heads_departures.notna().all().all())
assert(tails_arrivals.notna().all().all())

In [27]:
# fill first arrival, last departure with midnight the day before/after
first_arrival_update = sql_stop[['passservice_num', 'arrival']]
first_arrival_update = first_arrival_update.sort_values('arrival', na_position='first')
first_arrival_update['arrival'] = first_arrival_update['arrival'].dt.floor('D')
first_arrival_update['arrival'] = first_arrival_update.groupby('passservice_num')['arrival'].bfill(limit=1)
first_arrival_update = first_arrival_update.groupby('passservice_num').nth(0)
sql_stop['arrival'] = first_arrival_update['arrival'].reindex_like(sql_stop).where(pd.notna, sql_stop['arrival'])

last_departure_update = sql_stop[['passservice_num', 'arrival', 'departure']]
last_departure_update = last_departure_update.sort_values('arrival', na_position='first')
last_departure_update['departure'] = last_departure_update['departure'].dt.ceil('D')
last_departure_update['departure'] = last_departure_update.groupby('passservice_num')['departure'].ffill(limit=1)
last_departure_update = last_departure_update.groupby('passservice_num').nth(-1)
sql_stop['departure'] = last_departure_update['departure'].reindex_like(sql_stop).where(pd.notna, sql_stop['departure'])

In [28]:
# double-check table keys
assert (set(sql_passservice['trainset']) == set(sql_trainsetamenities['trainset']))
assert (set(sql_stop['stations_uic']) < set(sql_stations['uic']))
assert (set(sql_stop['passservice_num']) == set(sql_passservice['num']))

AssertionError: 

In [ ]:
# also check which columns of any tables are nullable
assert (sql_stop.notna().all().all())
assert (sql_stations.notna().all().all())
assert (sql_passservice.notna().all().all())
assert (sql_trainsetamenities.notna().all().all())
# print('sql_stop', sql_stop.notna().all(axis='index'))
# print('sql_stations', sql_stations.notna().all(axis='index'))
# print('sql_passservice', sql_passservice.notna().all(axis='index'))
# print('sql_trainsetamenities', sql_trainsetamenities.notna().all(axis='index'))

## Export

In [ ]:
# export with .to_csv()
sql_stations.to_csv(out_dir / 'stations.csv')
sql_trainsetamenities.to_csv(out_dir / 'trainsetamenities.csv')
sql_passservice.to_csv(out_dir / 'passservice.csv')
sql_stop.to_csv(out_dir / 'stop.csv')

# UNUSED BELOW

## Get train stock

In [ ]:
# TREIN '/v1/trein' https://apiportal.ns.nl/api-details#api=virtual-train-API&operation=getTrainInformation
# TREINRIT '/v1/trein/{ritnummer}' https://apiportal.ns.nl/api-details#api=virtual-train-API&operation=getTrainInformationForRitnummer
# STATION VAN TREINRIT '/v1/trein/{ritnummer}/{stationscode}' https://apiportal.ns.nl/api-details#api=virtual-train-API&operation=getTrainInformationForRitnummerAndStationCode

# (Below are available from the three APIs above, but not all are applicable without specifying Rit and/or Station.)
# (Één MaterieelDeel bestaat uit meerdere Bakken.)

# TreinInformatie.ritnummer (int)
# TreinInformatie.type (string)
# TreinInformatie.geplandeMaterieeldelen[].materieelType (string)
# TreinInformatie.geplandeMaterieeldelen[].materieelnummer (int)
# TreinInformatie.geplandeMaterieeldelen[].type (string)
# TreinInformatie.geplandeMaterieeldelen[].faciliteiten (string[])

In [ ]:
# trein_pkl = Path('./ns_results/raw/trein_info.pkl')
# if not trein_pkl.exists():
#     response = requests.get(NSAPI_URLS['VIRTUAL_TRAIN'] + '/v1' + '/trein', headers=NS_GET_HEADER,
#         params=nsapi.virtual_train.ApiV1TreinGetParameters(
#             ids=train_id,
#             # dateTime=(datetime.today() + timedelta(days=1)).isoformat().__str__(),
#             all=True,
#         )
#     )
#     print('Response:', response.json())
#     obj = nsapi.virtual_train.TreinInformatie(**response.json())
#     with open(trein_pkl, 'wb') as file:
#         pickle.dump(obj, file, protocol=-1)
# else:
#     with open(trein_pkl, 'rb') as file:
#         obj = pickle.load(file)

In [ ]:
# do we want to do /trein/ritnummer?
# trein_rit_pkl = Path('./ns_results/raw/trein_met_ritnummer_info.pkl')
# if not trein_rit_pkl.exists():
#     response = requests.get(NSAPI_URLS['VIRTUAL_TRAIN'] + '/v1' + f'/trein/{ritnummer}', headers=NS_GET_HEADER,
#         params=nsapi.virtual_train.ApiV1TreinRitnummerGetParameters(
#             # countryCodes=['NL'],
#             # limit=None # irrelevant when not providing str `q`
#         )
#     )
#     obj = nsapi.virtual_train.TreinInformatie(**response.json())
#     with open(trein_rit_pkl, 'wb') as file:
#         pickle.dump(obj, file, protocol=-1)
# else:
#     with open(trein_rit_pkl, 'rb') as file:
#         obj = pickle.load(file)

In [ ]:
# do we want to do /trein/ritnummer/stationscode?
# trein_rit_stations_pkl = Path('./ns_results/raw/trein_met_ritnummer_en_stationscode_info.pkl')
# if not trein_rit_stations_pkl.exists():
#     response = requests.get(NSAPI_URLS['VIRTUAL_TRAIN'] + '/v1' + f'/trein/{ritnummer}/{stationscode}', headers=NS_GET_HEADER,
#         params=nsapi.virtual_train.ApiV1TreinRitnummerStationscodeGetParameters(
#             # countryCodes=['NL'],
#             # limit=None # irrelevant when not providing str `q`
#         )
#     )
#     obj = nsapi.virtual_train.TreinInformatie(**response.json())
#     with open(trein_rit_stations_pkl, 'wb') as file:
#         pickle.dump(obj, file, protocol=-1)
# else:
#     with open(trein_rit_stations_pkl, 'rb') as file:
#         obj = pickle.load(file)

## Other data

In [ ]:
# ARRIVALS 'https://gateway.apiportal.ns.nl/timetable-api/v2/arrivals' https://apiportal.ns.nl/api-details#api=reisinformatie-api&operation=getArrivals&definition=Arrival
# DEPARTURES 'https://gateway.apiportal.ns.nl/timetable-api/v2/departures' https://apiportal.ns.nl/api-details#api=reisinformatie-api&operation=getDepartures
# Arrivals[].plannedTimeZoneOffset
# Arrivals[].plannedDateTime
# Arrivals[].name
# Arrivals[].trainCategory
#   Arrivals[].product.displayName
#   Arrivals[].product.shortCategoryName
# Arrivals[].product.number (string)

In [ ]:
# arrivals_pkl = Path('./ns_results/raw/arrivals.pkl')
# if not arrivals_pkl.exists():
#     response = requests.get('https://gateway.apiportal.ns.nl/timetable-api' + '/v2' + '/arrivals', headers=NS_GET_HEADER,
#         params=nsapi.reisinformatie.ApiV2ArrivalsGetParameters(
#             # uicCode=station_uic_code
#             maxJourneys=100 # default 40
#         )
#     )
#     obj = nsapi.reisinformatie.RepresentationResponseArrivalsPayload(**response.json())
#     with open(arrivals_pkl, 'wb') as file:
#         pickle.dump(obj, file, protocol=-1)
# else:
#     with open(arrivals_pkl, 'rb') as file:
#         obj = pickle.load(file)

In [ ]:
# departures_pkl = Path('./ns_results/raw/departures.pkl')
# if not departures_pkl.exists():
#     response = requests.get('https://gateway.apiportal.ns.nl/timetable-api' + '/v2' + '/departures', headers=NS_GET_HEADER,
#         params=nsapi.reisinformatie.ApiV2DeparturesGetParameters(
#             # uicCode=station_uic_code
#             maxJourneys=100, # default 40
#         )
#     )
#     obj = nsapi.reisinformatie.RepresentationResponseDeparturesPayload(**response.json())
#     with open(departures_pkl, 'wb') as file:
#         pickle.dump(obj, file, protocol=-1)
# else:
#     with open(departures_pkl, 'rb') as file:
#         obj = pickle.load(file)